# Build Player Career Stats - Silver Layer

Aggregates StatsBomb event data from bronze layer to create player career statistics.

## Target Schema
* **player_id** (INT, PK)
* **player_name** (STRING)
* **total_matches** (INT)
* **total_shots** (INT)
* **total_goals** (INT)
* **total_xg** (DOUBLE)
* **avg_xg_per_shot** (DOUBLE)
* **shot_conversion_pct** (DOUBLE)
* **total_pressures** (INT)
* **total_dribbles** (INT)
* **last_updated** (TIMESTAMP)

## Process
1. Read events from bronze layer
2. Extract shot metrics from raw_json
3. Aggregate by player
4. Calculate derived metrics
5. Display top performers

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from config.paths import EVENTS_BRONZE

# Initialize Spark session
spark = SparkSession.builder.appName("BuildPlayerCareerStats").getOrCreate()

print("=" * 80)
print("Building Player Career Stats - Silver Layer")
print("=" * 80)

In [0]:
# Read events from bronze layer
print("\n[1/5] Reading events from bronze layer...")
df_events = spark.read.parquet(EVENTS_BRONZE)
print(f"   Total events loaded: {df_events.count():,}")

# Filter for players only (exclude NULL player_id)
df_player_events = df_events.filter(F.col("player_id").isNotNull())
print(f"   Player events: {df_player_events.count():,}")

In [0]:
# Parse raw_json to extract shot-specific fields
print("\n[2/5] Extracting shot metrics from raw_json...")
df_with_shot_data = df_player_events.withColumn(
    "shot_xg",
    F.when(
        F.col("event_type_name") == "Shot",
        F.get_json_object(F.col("raw_json"), "$.shot.statsbomb_xg").cast(DoubleType())
    ).otherwise(F.lit(None))
).withColumn(
    "shot_outcome",
    F.when(
        F.col("event_type_name") == "Shot",
        F.get_json_object(F.col("raw_json"), "$.shot.outcome.name")
    ).otherwise(F.lit(None))
)

print("   Shot data extracted ")

In [0]:
display(df_with_shot_data.limit(30))

In [0]:
# Aggregate by player
print("\n[3/5] Aggregating player statistics...")
df_player_stats = df_with_shot_data.groupBy("player_id", "player_name").agg(
    # Match participation
    F.countDistinct("match_id").alias("total_matches"),
    
    # Shot metrics
    F.sum(F.when(F.col("event_type_name") == "Shot", 1).otherwise(0)).alias("total_shots"),
    F.sum(F.when(F.col("shot_outcome") == "Goal", 1).otherwise(0)).alias("total_goals"),
    F.sum(F.coalesce(F.col("shot_xg"), F.lit(0.0))).alias("total_xg"),
    
    # Other event metrics
    F.sum(F.when(F.col("event_type_name") == "Pressure", 1).otherwise(0)).alias("total_pressures"),
    F.sum(F.when(F.col("event_type_name") == "Dribble", 1).otherwise(0)).alias("total_dribbles")
)

print("   Aggregation complete ")

In [0]:
display(df_player_stats.limit(49))

In [0]:
df_player_stats.count()

In [0]:
# Calculate derived metrics
print("\n[4/5] Calculating derived metrics...")
df_final = df_player_stats.withColumn(
    "avg_xg_per_shot",
    F.when(
        F.col("total_shots") > 0,
        F.col("total_xg") / F.col("total_shots")
    ).otherwise(0.0)
).withColumn(
    "shot_conversion_pct",
    F.when(
        F.col("total_shots") > 0,
        (F.col("total_goals") / F.col("total_shots")) * 100
    ).otherwise(0.0)
).withColumn(
    "last_updated",
    F.current_timestamp()
).select(
    F.col("player_id").cast("int"),
    F.col("player_name"),
    F.col("total_matches").cast("int"),
    F.col("total_shots").cast("int"),
    F.col("total_goals").cast("int"),
    F.col("total_xg"),
    F.col("avg_xg_per_shot"),
    F.col("shot_conversion_pct"),
    F.col("total_pressures").cast("int"),
    F.col("total_dribbles").cast("int"),
    F.col("last_updated")
)

print("   Derived metrics calculated ✓")

In [0]:
display(df_final.limit(50))

In [0]:
# Show summary statistics
print("\n[5/5] Summary Statistics:")
print(f"   Total players: {df_final.count():,}")
print(f"   Players with shots: {df_final.filter(F.col('total_shots') > 0).count():,}")
print(f"   Players with goals: {df_final.filter(F.col('total_goals') > 0).count():,}")

print("\nTop 10 Goal Scorers:")
df_final.orderBy(F.desc("total_goals")).select(
    "player_name", "total_matches", "total_shots", "total_goals", 
    "total_xg", "shot_conversion_pct"
).show(10, truncate=False)

print("\nTop 10 by xG:")
df_final.orderBy(F.desc("total_xg")).select(
    "player_name", "total_shots", "total_goals", "total_xg", "avg_xg_per_shot"
).show(10, truncate=False)

# Write to silver layer
print("\n[OUTPUT] Writing to matchpulse.silver.player_career_stats...")
df_final.write.mode("overwrite").saveAsTable("matchpulse.silver.player_career_stats")
print("✓ Successfully wrote to matchpulse.silver.player_career_stats")

print("\n" + "=" * 80)
print("Build Complete")
print("=" * 80)

In [0]:
%sql
-- Verify the table was created successfully
DESCRIBE TABLE EXTENDED matchpulse.silver.player_career_stats;

In [0]:
%sql
-- Query sample data from the table
SELECT 
    player_name,
    total_matches,
    total_shots,
    total_goals,
    ROUND(total_xg, 2) as total_xg,
    ROUND(avg_xg_per_shot, 3) as avg_xg_per_shot,
    ROUND(shot_conversion_pct, 1) as shot_conv_pct,
    total_pressures,
    total_dribbles
FROM matchpulse.silver.player_career_stats
ORDER BY total_goals DESC
LIMIT 30;

In [0]:
# Display sample of final dataset
print("Sample records with all columns:")
display(df_final.orderBy(F.desc("total_goals")).limit(10))

In [0]:
# =========================================
# Imports
# =========================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F

sns.set_style("whitegrid")

# =========================================
# Read Silver Table
# =========================================

df = spark.table("matchpulse.silver.player_career_stats")

display(df.limit(5))

In [0]:
df_metrics = (
    df
    .withColumn(
        "finishing_diff",
        F.round(
            F.col("total_goals") - F.col("total_xg"),
            2
        )
    )
    .withColumn(
        "goal_per_xg",
        F.round(
            F.try_divide(
                F.col("total_goals"),
                F.col("total_xg")
            ),
            2
        )
    )
)

In [0]:
display(df_metrics.limit(5))

In [0]:
df_filtered = (
    df_metrics
    .filter(F.col("total_xg") >= 5)
    .filter(F.col("total_goals") >= 5)
)

In [0]:
pdf = df_filtered.select(
    "player_name",
    "total_goals",
    "total_xg",
    "finishing_diff",
    "goal_per_xg"
).toPandas()

In [0]:
# Sort players
pdf_sorted = pdf.sort_values(
    "finishing_diff",
    ascending=False
)

# Top & bottom
top_over = pdf_sorted.head(10)
top_under = pdf_sorted.tail(10)

plot_df = pd.concat([top_over, top_under])

# Plot
plt.figure(figsize=(14, 8))

sns.barplot(
    data=plot_df,
    x="finishing_diff",
    y="player_name",
    palette="RdYlGn"
)

plt.axvline(0, linestyle='--', color='black')

plt.title(
    "Top xG Overperformers & Underperformers",
    fontsize=18
)

plt.xlabel("Goals - xG")
plt.ylabel("Player")

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(12, 10))

sns.scatterplot(
    data=pdf,
    x="total_xg",
    y="total_goals",
    hue="finishing_diff",
    size="total_goals",
    palette="coolwarm",
    sizes=(50, 400),
    alpha=0.8
)

# Perfect finishing line
max_val = max(
    pdf["total_xg"].max(),
    pdf["total_goals"].max()
)

plt.plot(
    [0, max_val],
    [0, max_val],
    linestyle='--',
    color='black'
)

# Annotate top overperformers
for _, row in pdf.nlargest(8, "finishing_diff").iterrows():

    plt.text(
        row["total_xg"] + 0.1,
        row["total_goals"] + 0.1,
        row["player_name"],
        fontsize=9
    )

plt.title("Goals vs xG", fontsize=18)

plt.xlabel("Total xG")
plt.ylabel("Total Goals")

plt.tight_layout()
plt.show()

In [0]:
# Keep only players above threshold
pdf_filtered = pdf[pdf["total_goals"] >= 30]

# Sort after filtering
clinical = (
    pdf_filtered
    .sort_values("goal_per_xg", ascending=False)
    .head(15)
)

# Plot
plt.figure(figsize=(12, 8))

sns.barplot(
    data=clinical,
    x="goal_per_xg",
    y="player_name",
    palette="viridis"
)

plt.title(
    "Most Clinical Finishers (30+ Goals)",
    fontsize=18
)

plt.xlabel("Goals / xG")
plt.ylabel("Player")

plt.tight_layout()
plt.show()